# model_test — 부스팅 XGBoost (Colab GPU)

v2 모델 비교의 XGBoost 3종: 회귀, pairwise, 순위 회귀. 설계는 `config/experiment.yaml`의 `MODEL_GRID`와 노션 「11주차 v2 모델」. 계산 코드는 `src/models/boosting.py`·`src/evaluation/model_test.py`, 이 노트북은 함수만 부른다.

- **장치**: GPU(cuda). 고를 때 쓴 장치로 최종 모델도 만든다(2026-11-14 결정). GPU 런타임이 아니면 1번 셀이 멈춘다.
- **Colab 준비**: 런타임 유형 GPU → 아래 설치 줄 실행 → 런타임 다시 시작 → 0번부터.
  `!pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3`
  (google-colab·numba·cudf 버전 경고와 bottleneck 경고는 무시해도 된다.)
- **이어서 돌리기 없음**: 종류마다 계산 직후 검사·저장한다. 세션이 끊기면 0번부터 다시 하고, 이미 저장된 종류는 건너뛴다.
- 이번 단계는 개별형만. 지수형(① → ②)은 개별형 결과를 본 뒤 셀을 추가한다.

## 0. 실행 환경 확인

계산은 하지 않고 위치·GPU·버전만 확인한다.

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

In [ ]:
import os, subprocess
print('CPU 개수:', os.cpu_count())
if os.name != 'nt':   # Linux(Colab)에서만
    print(subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
    print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)

## 1. 경로·설정

- **Colab**: `COMMIT`에 실행할 커밋 해시를 넣는다. 이미 받은 저장소 폴더도 `fetch` 후 그 커밋으로 맞춘다. `data/`는 Drive 공유 폴더로 연결.
- **로컬**: 이 저장소를 그대로 쓴다. 결과는 `outputs/run_id/model_test/<시각>_<종류>_individual/`.
- 장치는 config `MODEL_GRID.device`를 따른다(XGBoost cuda, LightGBM cpu).

In [ ]:
import os, sys, re, subprocess
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
COMMIT = '9d259427f2c998fb434f96b632b55cc62660a88e'
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not COMMIT:
        raise RuntimeError('COMMIT을 채워야 Colab에서 실행할 수 있다(재현성: 커밋 고정).')
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin'], check=True)   # 이미 받은 폴더도 새 커밋을 알게 함
    subprocess.run(['git', '-C', str(ROOT), 'checkout', COMMIT], check=True)
    if not (ROOT / 'data').is_symlink():       # 저장소의 data/(.gitkeep만 있음)를 Drive 공유 data로 바꿔 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = DRIVE_BASE / 'runs' / 'model_test'
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None    # 기본: ROOT/outputs/run_id/model_test

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_grid(config)

# requirements.txt 고정 버전과 설치 버전 비교(다르면 경고)
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML', 'xgboost', 'lightgbm']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")

started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT   :', ROOT)
import shutil
from src.models import boosting
GPU_NAME = (subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
            if shutil.which('nvidia-smi') else None)
if not GPU_NAME:
    raise RuntimeError('XGBoost는 config상 GPU(cuda)로 고르고 최종 모델도 GPU다. GPU 런타임으로 바꿔 실행한다.')
N_THREADS = os.cpu_count()
print('저장   :', OUT_ROOT or ROOT / config['PATHS']['run_records'] / 'model_test')
print('GPU    :', GPU_NAME, '| 스레드:', N_THREADS, '| 시작:', started.isoformat(timespec='seconds'))

## 2. 입력 준비 (약 2분)

λ 후보 10개의 σ를 수익률로 다시 계산하고(λ=0.97 값이 `h3.parquet`와 정확히 같은지 검사), 매달 순위 [−1, 1] 입력과 목표값 3종·순위 정답 2종(10분위, 백분위)(그 달 학습 행 기준)을 만든다. 확인할 값: 검증 예측 133,876행(모두 적격), 학습 행 2003년~ 151,761행.

In [ ]:
import time
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config)
print(f'입력 준비 {time.perf_counter() - t0:.0f}초')
for k, v in prepared['checks'].items():
    print(f'  {k:45s} {v}')
print('λ 후보:')
print(prepared['lambda_candidates'].round(4).to_string(index=False))
results, saved = {}, {}   # 종류별 결과·저장 위치를 이 세션 메모리에 모은다

## 3. XGBoost 회귀 — 개별형 [Size, B/M, Mom, σ] (예상 약 1.1시간)

설정 12개 × Fold 16 × H3 λ 3개(span 12·36·60). 학습 한 번마다 한 줄(고른 트리 수, 학습 IC, 내부 검증 IC, 시간), 설정마다 선택 줄. 끝나면 종료 조건을 검사하고 바로 저장한다(실패하면 표를 쓰지 않음).

In [ ]:
t0 = time.perf_counter()
results['xgb_reg'] = mt.run_boosting(prepared, config, 'xgb_reg', phase='individual', n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_reg'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_reg'], started=started,
                                       out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_reg']['status'], saved['xgb_reg']['run_dir'])
mt.boosting_part_summary(results['xgb_reg'])

## 4. XGBoost pairwise — 개별형 [Size, B/M, Mom, σ] (예상 약 1.2시간)

설정 3개 × Fold 16 × H3 λ 3개(span 12·36·60). 학습 한 번마다 한 줄(고른 트리 수, 학습 IC, 내부 검증 IC, 시간), 설정마다 선택 줄. 끝나면 종료 조건을 검사하고 바로 저장한다(실패하면 표를 쓰지 않음).

In [ ]:
t0 = time.perf_counter()
results['xgb_pairwise'] = mt.run_boosting(prepared, config, 'xgb_pairwise', phase='individual', n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_pairwise'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_pairwise'], started=started,
                                       out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_pairwise']['status'], saved['xgb_pairwise']['run_dir'])
mt.boosting_part_summary(results['xgb_pairwise'])

## 5. XGBoost 순위 회귀 — 개별형 [Size, B/M, Mom, σ] (예상 약 0.3시간)

설정 3개 × Fold 16 × H3 λ 3개(span 12·36·60). 학습 한 번마다 한 줄(고른 트리 수, 학습 IC, 내부 검증 IC, 시간), 설정마다 선택 줄. 끝나면 종료 조건을 검사하고 바로 저장한다(실패하면 표를 쓰지 않음).

In [ ]:
t0 = time.perf_counter()
results['xgb_rankreg'] = mt.run_boosting(prepared, config, 'xgb_rankreg', phase='individual', n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_rankreg'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_rankreg'], started=started,
                                       out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_rankreg']['status'], saved['xgb_rankreg']['run_dir'])
mt.boosting_part_summary(results['xgb_rankreg'])

## 6. 후속 튜닝 — XGBoost 순위 회귀 깊이 4 (예상 약 10분)

순위 정답 3종은 깊이 1→2→3에서 IC가 계속 올라 깊이 4를 한 번 추가로 시험한다(2026-11-14 결정, config `boosting_followup_depths`). 설정 1개 × Fold 16 × H3 λ 3개. 원래 결과와 섞지 않고 `<시각>_<종류>_individual_d4` 폴더에 따로 저장한다. 위 셀과 같은 세션이면 0~2번을 다시 돌릴 필요 없다.

In [ ]:
t0 = time.perf_counter()
results['xgb_rankreg_d4'] = mt.run_boosting(prepared, config, 'xgb_rankreg', phase='individual', followup=True, n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_rankreg_d4'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_rankreg_d4'], started=started,
                                          out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_rankreg_d4']['status'], saved['xgb_rankreg_d4']['run_dir'])
mt.boosting_part_summary(results['xgb_rankreg_d4'])

## 7. 후속 튜닝 — XGBoost pairwise 깊이 4 (예상 약 15분)

순위 정답 3종은 깊이 1→2→3에서 IC가 계속 올라 깊이 4를 한 번 추가로 시험한다(2026-11-14 결정, config `boosting_followup_depths`). 설정 1개 × Fold 16 × H3 λ 3개. 원래 결과와 섞지 않고 `<시각>_<종류>_individual_d4` 폴더에 따로 저장한다. 위 셀과 같은 세션이면 0~2번을 다시 돌릴 필요 없다.

In [ ]:
t0 = time.perf_counter()
results['xgb_pairwise_d4'] = mt.run_boosting(prepared, config, 'xgb_pairwise', phase='individual', followup=True, n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_pairwise_d4'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_pairwise_d4'], started=started,
                                          out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_pairwise_d4']['status'], saved['xgb_pairwise_d4']['run_dir'])
mt.boosting_part_summary(results['xgb_pairwise_d4'])

# 지수형 (① H1 지수 → ② 모델)

- 위 개별형·깊이 4 셀과 상관없다. 새 세션이면 **0 → 1 → 2번만** 돌린 뒤 아래 셀로 온다(개별형 셀은 건너뜀).
- ① 설정마다 2003~2021년 H1 지수(Size·B/M·Mom → 순위 [−1, 1])를 만든다. σ를 쓰지 않아 λ는 고르지 않고 트리 수만 고른다.
- ② 입력 3가지 [σ]·[H1 지수]·[H1 지수, σ]. λ(span 12·36·60)·트리 수·동점 규칙·L2 0·깊이 1~3은 개별형과 같다.
- ①은 같은 세션 메모리로 ②에 넘긴다(파일로 저장했다 다시 읽지 않음). ②가 실패하면 ② 셀만 다시 돌리면 된다.
- 종류마다 ② 직후 검사·저장, 폴더 `<시각>_<종류>_index`. 예상 시간은 개별형 실측 × 약 2.7배(입력이 적어 실제는 더 짧을 수 있음).

## 8. 지수형 — XGBoost 순위 회귀 (예상 약 0.8시간)

In [ ]:
t0 = time.perf_counter()
stage1_xgb_rankreg = mt.run_boosting_stage1(prepared, config, 'xgb_rankreg', n_threads=N_THREADS)
print(f'지수형 ① {(time.perf_counter() - t0) / 60:.1f}분')

In [ ]:
t0 = time.perf_counter()
results['xgb_rankreg_index'] = mt.run_boosting(prepared, config, 'xgb_rankreg', phase='index', stage1=stage1_xgb_rankreg,
                                        n_threads=N_THREADS)
print(f'지수형 ② {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_rankreg_index'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_rankreg_index'], started=started,
                                             out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_rankreg_index']['status'], saved['xgb_rankreg_index']['run_dir'])
mt.boosting_part_summary(results['xgb_rankreg_index'])

## 9. 지수형 — XGBoost pairwise (예상 약 3시간)

In [ ]:
t0 = time.perf_counter()
stage1_xgb_pairwise = mt.run_boosting_stage1(prepared, config, 'xgb_pairwise', n_threads=N_THREADS)
print(f'지수형 ① {(time.perf_counter() - t0) / 60:.1f}분')

In [ ]:
t0 = time.perf_counter()
results['xgb_pairwise_index'] = mt.run_boosting(prepared, config, 'xgb_pairwise', phase='index', stage1=stage1_xgb_pairwise,
                                        n_threads=N_THREADS)
print(f'지수형 ② {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_pairwise_index'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_pairwise_index'], started=started,
                                             out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_pairwise_index']['status'], saved['xgb_pairwise_index']['run_dir'])
mt.boosting_part_summary(results['xgb_pairwise_index'])

## 10. 지수형 — XGBoost 회귀 (예상 약 3시간)

In [ ]:
t0 = time.perf_counter()
stage1_xgb_reg = mt.run_boosting_stage1(prepared, config, 'xgb_reg', n_threads=N_THREADS)
print(f'지수형 ① {(time.perf_counter() - t0) / 60:.1f}분')

In [ ]:
t0 = time.perf_counter()
results['xgb_reg_index'] = mt.run_boosting(prepared, config, 'xgb_reg', phase='index', stage1=stage1_xgb_reg,
                                        n_threads=N_THREADS)
print(f'지수형 ② {(time.perf_counter() - t0) / 60:.1f}분')
saved['xgb_reg_index'] = mt.finalize_boosting(ROOT, config, prepared, results['xgb_reg_index'], started=started,
                                             out_root=OUT_ROOT, gpu_name=GPU_NAME)
print(saved['xgb_reg_index']['status'], saved['xgb_reg_index']['run_dir'])
mt.boosting_part_summary(results['xgb_reg_index'])